In [1]:
#import packages to run GEE python API

import ee
import geemap
import pprint
import ipyleaflet
import time

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library
ee.Initialize(project='ee-diptibaral21')

In [2]:
#load packages
import os
import numpy as np
import pandas as pd

In [3]:
# STEP 0: Setup and Counties

# From TIGER2018 filter for 9 counties in sacramento valley

In [4]:

counties = ee.FeatureCollection("TIGER/2018/Counties")
selected_counties = ['Butte','Colusa','Glenn','Placer','Sacramento','San Joaquin','Sutter','Yolo','Yuba']

sac_valley = counties.filter(ee.Filter.eq('STATEFP', '06')) \
                     .filter(ee.Filter.inList('NAME', selected_counties))


In [5]:
#visualizing Sac Valley Counties

Map = geemap.Map()
Map.centerObject(sac_valley, 8)  

# Add the layer
Map.addLayer(sac_valley, {},'Sacramento Valley Counties')
Map

Map(center=[39.00285080659931, -121.59337946044376], controls=(WidgetControl(options=['position', 'transparent…

In [6]:
# STEP 1: Rice Masking


In [7]:
# helper function to mask rice -- rice is represented by cropland value 3

def mask_rice(image):
    rice = image.select('cropland').eq(3)
    return image.updateMask(rice) \
    .clip(sac_valley) \
    .unmask(0) \
    .set('system:time_start', image.get('system:time_start'))

# hepler function to make rice frequency 
# all those pixels that were once a rice growing area in last 45 years is counted as rice growing areas 
def get_rice_frequency(start_year, end_year):
    cdl = ee.ImageCollection("USDA/NASS/CDL") \
            .filterDate(f'{start_year}-01-01', f'{end_year}-12-31') \
            .map(lambda img: img.select(['cropland'])) \
            .map(mask_rice)

    rice_sum = cdl.reduce(ee.Reducer.sum())
    rice_frequency = rice_sum.gt(0)
    return rice_sum, rice_frequency
    
rice_sum, rice_frequency = get_rice_frequency(1997, 2023)   
Map.addLayer(rice_sum.clip(sac_valley),{'min':0, 'max': 27, 'palette': ['#e5f5e0', '#a1d99b', '#31a354']},'Rice Frequency (Years)')
Map



Map(bottom=25346.0, center=[39.00285080659931, -121.59337946044376], controls=(WidgetControl(options=['positio…

In [8]:
# STEP 2: Processing gridMET Data


In [9]:

def load_preprocess_gridMET(start_date, end_date):
    # filtering for tmmn and tmmx
    gridMET = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET") \
                 .filterDate(start_date, end_date) \
                 .select(['tmmx','tmmn'])
    # converting K to C
    def convert_units(img):
        tmmxC = img.select('tmmx').subtract(273.15).rename('tmmx')
        tmmnC = img.select('tmmn').subtract(273.15).rename('tmmn')
        tmeanC = tmmxC.add(tmmnC).divide(2).rename('tmean')
        return img.addBands([tmmxC, tmmnC, tmeanC], overwrite=True) \
                  .copyProperties(img, ['system:time_start'])

    return gridMET.map(convert_units).map(lambda img: img.updateMask(rice_frequency))

# filtering the date range from 1979 to 2023
gridMET_processed = load_preprocess_gridMET('1979-01-01','2023-12-31')

first_image = gridMET_processed.first()
info = first_image.getInfo()
#pprint.pprint(info)


In [10]:
# visualization of gridMET
Map.addLayer(
    gridMET_processed,
     {},
    'gridMET example image (Years)'
    )
#Map.split_map(left_layer='rice_frequency', right_layer='rice_sum')
Map

Map(bottom=25346.0, center=[39.00285080659931, -121.59337946044376], controls=(WidgetControl(options=['positio…

In [20]:
#STEP 3: Processing USDA Progress Data

# interpolating for planting and harvesting date


In [11]:
#STEP 4: Insert planting-harvest table

In [12]:
# Load planting-harvest table
season_table = ee.FeatureCollection('projects/ee-diptibaral21/assets/Planting_Harvest_Dates')
#data processing script(planting_harvest.ipynb)

In [13]:
# Build growing season collection
def filter_by_season(feature, initial):
    plant_date = ee.Date(feature.get('date_planting'))
    harvest_date = ee.Date(feature.get('date_harvest'))
    year = ee.Number(feature.get('year'))

    filtered = gridMET_processed.filterDate(plant_date, harvest_date) \
                                .map(lambda img: img.set('year', year))

    return ee.ImageCollection(initial).merge(filtered)

growing_season_images = ee.ImageCollection(season_table.iterate(filter_by_season, ee.ImageCollection([])))

In [14]:
# Extract daily stats per county
def extract_daily_stats(image):
    date = ee.Date(image.get('system:time_start'))
    reduced = image.reduceRegions(collection=sac_valley,
                                  reducer=ee.Reducer.mean(),
                                  scale=4000,
                                  crs='EPSG:3310')

    def set_props(f):
        return f.set({
            'date': date.format('YYYY-MM-dd'),
            'year': date.get('year'),
            'month': date.get('month'),
            'day': date.get('day')
        })

    return reduced.map(set_props)

daily_stats = growing_season_images.map(extract_daily_stats).flatten()

In [15]:
# Format results for export
formatted_stats = daily_stats.map(lambda f: ee.Feature(None, {
    'county': f.get('NAME'),
    'date': f.get('date'),
    'year': f.get('year'),
    'month': f.get('month'),
    'day': f.get('day'),
    'tmmn': f.get('tmmn'),
    'tmmx': f.get('tmmx'),
    'tmean': f.get('tmean')
}))


In [17]:
# Export 
task = ee.batch.Export.table.toDrive(
    collection=formatted_stats,
    description='Daily_Rice_Temperature_1979_2023',
    fileFormat='CSV',
    selectors=['county', 'date', 'year', 'month', 'day', 'tmmn', 'tmmx', 'tmean']
)
task.start()

In [18]:
# STEP 5: Phenology Detection Using Growing Degree Day Model - script (phenology_detection_gdd)

In [19]:
# STEP 6: Stress Calculation

# pixel wise stress calculation and then aggregaring over the couties


In [20]:

# Load stage table
stage_table = ee.FeatureCollection(
    "projects/ee-diptibaral21/assets/9_County_Stage_Dates_1979_2023"
)

In [21]:

# Function: calculate daily stress for a stage

def calculate_daily_stress(img, stage_feature):
    stage = ee.String(stage_feature.get('stage'))

    booting_cold = ee.Image.constant(13)
    flowering_cold = ee.Image.constant(10.9)
    flowering_heat = ee.Image.constant(37.5)

    # Default zero images
    cdstress_bo = ee.Image.constant(0).rename('cdstress_bo')
    cdstress_fl = ee.Image.constant(0).rename('cdstress_fl')
    htstress_fl = ee.Image.constant(0).rename('htstress_fl')

    # Cold stress for Booting stage
    cdstress_bo = ee.Image(
        ee.Algorithms.If(
            stage.equals('booting'),
            img.select('tmmn').lt(booting_cold)
               .multiply(booting_cold.subtract(img.select('tmmn')))
               .rename('cdstress_bo'),
            cdstress_bo
        )
    )

    # Cold stress for Flowering stage
    cdstress_fl = ee.Image(
        ee.Algorithms.If(
            stage.equals('flowering'),
            img.select('tmmn').lt(flowering_cold)
               .multiply(flowering_cold.subtract(img.select('tmmn')))
               .rename('cdstress_fl'),
            cdstress_fl
        )
    )

    # Heat stress for Flowering stage
    htstress_fl = ee.Image(
        ee.Algorithms.If(
            stage.equals('flowering'),
            img.select('tmmx').gt(flowering_heat)
               .multiply(img.select('tmmx').subtract(flowering_heat))
               .rename('htstress_fl'),
            htstress_fl
        )
    )

    return img.select(['tmmn', 'tmmx', 'tmean']).addBands([cdstress_bo, cdstress_fl, htstress_fl])




In [22]:
# ---------------------------------------------
# Debug daily stress
# ---------------------------------------------
first_stage_feature = stage_table.first()
example_date = ee.Date(first_stage_feature.get('start_date'))
example_temp_img = growing_season_images.filterDate(example_date, example_date.advance(1, 'day')).first()
example_daily_stress = calculate_daily_stress(example_temp_img, first_stage_feature)

print('Debug Daily Stress Function Results:', example_daily_stress.getInfo())

# Visualization
Map = geemap.Map(center=[38.5, -121.8], zoom=8)

Map.addLayer(
    example_daily_stress.select('cdstress_bo'),
    {'min': 0, 'max': 0.5, 'palette': ['white', 'yellow', 'red']},
    'Example Cold Stress Booting Viz'
)

Map.addLayer(
    example_daily_stress.select('tmmn'),
    {'min': 12, 'max': 15, 'palette': ['white', 'yellow', 'red']},
    'Example tmmn Viz'
)

Map.addLayer(
    example_daily_stress.select('tmmx'),
    {'min': 30, 'max': 33, 'palette': ['white', 'pink', 'red']},
    'Example tmmx Viz'
)
Map

Debug Daily Stress Function Results: {'type': 'Image', 'bands': [{'id': 'tmmn', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [1386, 585], 'crs': 'GEOGCS["unknown", \n  DATUM["unknown", \n    SPHEROID["Spheroid", 6378137.0, 298.257223563]], \n  PRIMEM["Greenwich", 0.0], \n  UNIT["degree", 0.017453292519943295], \n  AXIS["Longitude", EAST], \n  AXIS["Latitude", NORTH]]', 'crs_transform': [0.041666666666666664, 0, -124.78749996666667, 0, -0.041666666666666664, 49.42083333333334]}, {'id': 'tmmx', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [1386, 585], 'crs': 'GEOGCS["unknown", \n  DATUM["unknown", \n    SPHEROID["Spheroid", 6378137.0, 298.257223563]], \n  PRIMEM["Greenwich", 0.0], \n  UNIT["degree", 0.017453292519943295], \n  AXIS["Longitude", EAST], \n  AXIS["Latitude", NORTH]]', 'crs_transform': [0.041666666666666664, 0, -124.78749996666667, 0, -0.041666666666666664, 49.42083333333334]}, {'id': 'tmean', 'data_type': {'type': 'Pixe

Map(center=[38.5, -121.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [23]:


# Function: calculate yearly stress for one stage feature

def calculate_yearly_stage_stress(stage_feature):
    county = ee.String(stage_feature.get('county'))
    year = ee.Number(stage_feature.get('year'))
    start = ee.Date.parse('YYYY-MM-dd', stage_feature.get('start_date'))
    end = ee.Date.parse('YYYY-MM-dd', stage_feature.get('end_date'))

    stage_temps = growing_season_images.filterDate(start, end)

    # Map daily stress over images
    daily_stress = stage_temps.map(lambda img: calculate_daily_stress(img, stage_feature))

    # Mean of climate variables
    climate_means = daily_stress.select(['tmmn', 'tmmx', 'tmean']).mean()

    # Sum of stress indices
    stress_sums = daily_stress.select(['cdstress_bo', 'cdstress_fl', 'htstress_fl']).sum()

    # Combine
    yearly_stress = climate_means.addBands(stress_sums).set({
        'county': county,
        'stage': stage_feature.get('stage'),
        'year': year
    })

    # Reduce by county
    single_county = sac_valley.filter(ee.Filter.eq('NAME', county))
    return yearly_stress.reduceRegions(
        collection=single_county,
        reducer=ee.Reducer.mean(),
        scale=4000,
        crs='EPSG:3310'
    ).map(lambda f: f.set({
        'county': county,
        'stage': stage_feature.get('stage'),
        'year': year
    }))





In [24]:

# Debug yearly stage stress

debug_yearly_stage_table = stage_table \
    .filter(ee.Filter.eq('county', 'Butte')) \
    .filter(ee.Filter.eq('year', 2018)) \
    .filter(ee.Filter.eq('stage', 'flowering'))

print("Debug Stage Table:", debug_yearly_stage_table.getInfo())

yearly_stress_flowering_butte_2018 = calculate_yearly_stage_stress(debug_yearly_stage_table.first())
print("Yearly Stress during Flowering in Butte in 2018:", yearly_stress_flowering_butte_2018.getInfo())



Debug Stage Table: {'type': 'FeatureCollection', 'columns': {'county': 'String', 'end_date': 'String', 'stage': 'String', 'start_date': 'String', 'system:index': 'String', 'year': 'Integer'}, 'version': 1757889790414468, 'id': 'projects/ee-diptibaral21/assets/9_County_Stage_Dates_1979_2023', 'properties': {'system:asset_size': 10858}, 'features': [{'type': 'Feature', 'geometry': {'type': 'MultiPoint', 'coordinates': []}, 'id': '00000000000000000076', 'properties': {'county': 'Butte', 'end_date': '2018-07-28', 'stage': 'flowering', 'start_date': '2018-07-14', 'year': 2018}}]}
Yearly Stress during Flowering in Butte in 2018: {'type': 'FeatureCollection', 'columns': {'ALAND': 'Long', 'AWATER': 'Long', 'CBSAFP': 'String', 'CLASSFP': 'String', 'COUNTYFP': 'String', 'COUNTYNS': 'String', 'CSAFP': 'String', 'FUNCSTAT': 'String', 'GEOID': 'String', 'INTPTLAT': 'String', 'INTPTLON': 'String', 'LSAD': 'String', 'METDIVFP': 'String', 'MTFCC': 'String', 'NAME': 'String', 'NAMELSAD': 'String', 'STA

In [26]:

# Calculate stress for all years

all_year_table = ee.FeatureCollection(stage_table.map(calculate_yearly_stage_stress).flatten())

all_year_table_selected = all_year_table.select([
    'county', 'year', 'stage',
    'cdstress_bo', 'cdstress_fl', 'htstress_fl'
])

first_export_feature = all_year_table_selected.first()
first_export_feature.getInfo()


{'type': 'Feature',
 'geometry': {'type': 'Polygon',
  'coordinates': [[[-121.63636087086758, 39.245992618874666],
    [-121.6361813104536, 39.24522966403238],
    [-121.6361364855846, 39.245139877459195],
    [-121.63600183679381, 39.24496029189483],
    [-121.63595692481508, 39.244915463927676],
    [-121.63591201281854, 39.244780769790744],
    [-121.63577736376104, 39.24446659709374],
    [-121.63577736376104, 39.24442173529589],
    [-121.63568762664679, 39.24419732263753],
    [-121.6355978894615, 39.243883112883054],
    [-121.63555297732266, 39.24374848734932],
    [-121.63546324003073, 39.24338941214287],
    [-121.63541832783856, 39.24320985096277],
    [-121.63532859044, 39.24294056955423],
    [-121.63528367819445, 39.2428508044695],
    [-121.63528367819445, 39.2428059414132],
    [-121.63528367819445, 39.242761078326126],
    [-121.63528367819445, 39.24271614365622],
    [-121.63528367819445, 39.242626417328054],
    [-121.63523876593112, 39.24249171682906],
    [-121.635

In [27]:

# Export to Google Drive

task_drive = ee.batch.Export.table.toDrive(
    collection=all_year_table_selected,
    description='AnnualStress_1979_2023_v2',
    fileFormat='CSV',
    selectors=['county', 'year', 'stage', 'cdstress_bo', 'cdstress_fl', 'htstress_fl']
)
task_drive.start()


In [28]:

while task_drive.active():
    print('Status:', task_drive.status())
    time.sleep(5)

#print("Final status:", task_drive.status())
